# nb13 — Teacher Transformer Training (Kaggle / GPU)

Trains the **teacher** dense transformer on Tiny Shakespeare. Outputs are reused
by `nb14` (the student distillation notebook).

### What the teacher is for

The teacher is NOT what gets deployed to the 6502. It exists *only* to provide
**soft target distributions** for student training (knowledge distillation).

### Outputs

- `runs/teacher.pt` — model + config + history
- `runs/bpe.json` — BPE tokenizer (shared with student)

### Hyperparameter choices

- vocab=512 BPE (most common Shakespeare words are single tokens)
- `d_model=256`, `n_layers=6`, `num_heads=4`, `mlp_mult=4`
- ~3M params (real transformer scale, BPC target ~0.7–0.8)
- 15,000 steps (~10–15 min on a Kaggle T4/P100)


## Cell 1 — Setup (Kaggle environment)

Two ways to get `wozformer` available on Kaggle:

1. **Upload as a Kaggle dataset** — add the `wozformer/` directory + `pyproject.toml` + `data/tinyshakespeare.txt` as a dataset, then attach it. Modify `WOZFORMER_PATH` below.

2. **Git clone** — works if the repo is on GitHub. Modify `REPO_URL` below.


In [ ]:
import os, sys, subprocess
from pathlib import Path

# Option A: Kaggle dataset path (uncomment + adjust)
WOZFORMER_PATH = Path('/kaggle/input/wozformer')

# Option B: Git clone (uncomment + adjust)
# REPO_URL = 'https://github.com/elixpo/wozformer.git'
# subprocess.run(['git', 'clone', REPO_URL, '/kaggle/working/wozformer'], check=True)
# WOZFORMER_PATH = Path('/kaggle/working/wozformer')

# Install
if WOZFORMER_PATH.exists():
    os.chdir(WOZFORMER_PATH)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.'], check=True)
    print(f'wozformer installed from {WOZFORMER_PATH}')
else:
    print(f'WARNING: {WOZFORMER_PATH} not found. Adjust the path above.')

import wozformer as wz
print(f'wozformer version: {wz.__version__}')
print(f'device: {wz.utils.get_device()}')


## Cell 2 — Hyperparameters

Bigger than the local default — Kaggle gives you a T4 or P100 to spend.


In [ ]:
# Teacher hyperparameters
VOCAB_SIZE = 512
D_MODEL    = 256
NUM_HEADS  = 4
N_LAYERS   = 6
MLP_MULT   = 4
BLOCK_SIZE = 64

# Training
BATCH_SIZE = 64
LR         = 3e-4
N_STEPS    = 15000
EVAL_EVERY = 500
SEED       = 1337

# Outputs
OUT_DIR = Path('/kaggle/working/runs')
OUT_DIR.mkdir(parents=True, exist_ok=True)
TEACHER_PT = OUT_DIR / 'teacher.pt'
BPE_JSON   = OUT_DIR / 'bpe.json'


## Cell 3 — Load corpus + train BPE tokenizer

Tiny Shakespeare must be at `data/tinyshakespeare.txt` inside the dataset/clone.
BPE training is fast (~10 s).


In [ ]:
import torch

wz.utils.set_seed(SEED)
device = wz.utils.get_device()

# Find the corpus — try a few common locations
corpus_candidates = [
    WOZFORMER_PATH / 'data' / 'tinyshakespeare.txt',
    Path('/kaggle/input/tinyshakespeare/tinyshakespeare.txt'),
    Path('data/tinyshakespeare.txt'),
]
corpus_path = next((p for p in corpus_candidates if p.exists()), None)
assert corpus_path is not None, f'corpus not found in {corpus_candidates}'

text = wz.data.load_corpus(corpus_path)
tok  = wz.tokenizer.BPETokenizer.train(text, vocab_size=VOCAB_SIZE)
tok.save(BPE_JSON)
print(f'BPE saved → {BPE_JSON}')

ids = torch.tensor(tok.encode(text), dtype=torch.long)
train_data, val_data = wz.data.split_train_val(ids, val_fraction=0.1)
print(f'tokens: {len(ids):,}  (train {len(train_data):,} / val {len(val_data):,})')

# What did BPE learn? Peek at the longest 15 multi-char tokens.
long_toks = sorted([t for t in tok.itos if t != '<unk>' and not t.startswith('<pad')],
                   key=lambda x: -len(x))[:15]
print('longest BPE tokens:', long_toks)


## Cell 4 — Build and inspect the teacher


In [ ]:
tcfg = wz.config.TransformerConfig(
    vocab_size=VOCAB_SIZE,
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    n_layers=N_LAYERS,
    mlp_mult=MLP_MULT,
)
teacher = wz.models.TinyTransformer(tcfg, block_size=BLOCK_SIZE).to(device)

n_params = wz.utils.count_params(teacher)
print(f'teacher params: {n_params:,}  (~{n_params/1e6:.2f} M)')

# Sanity-check forward
x = torch.randint(0, VOCAB_SIZE, (4, BLOCK_SIZE), device=device)
y = torch.randint(0, VOCAB_SIZE, (4, BLOCK_SIZE), device=device)
with torch.no_grad():
    logits, loss = teacher(x, y)
import math
print(f'init loss: {loss.item():.4f}  (expect ~{math.log(VOCAB_SIZE):.4f} = ln(V))')


## Cell 5 — Train the teacher

~10–15 min on a Kaggle T4. The trainer prints `val(soft)` every 500 steps and saves the best checkpoint to RAM.

At step 0 you should see val ≈ ln(512) ≈ 6.24 (random baseline). Watch it descend.


In [ ]:
train_cfg = wz.config.TrainConfig(
    batch_size=BATCH_SIZE,
    block_size=BLOCK_SIZE,
    lr=LR,
    n_steps=N_STEPS,
    eval_every=EVAL_EVERY,
    seed=SEED,
)
history, best = wz.trainer.train(teacher, train_data, val_data, train_cfg, device=device)
print(f'\nbest val: {best["val"]:.4f} at step {best["step"]}')


## Cell 6 — Plot the loss curve and report BPC


In [ ]:
import matplotlib.pyplot as plt

steps   = [h[0] for h in history]
trains  = [h[1] for h in history]
vals    = [h[2] for h in history]

plt.figure(figsize=(9, 4))
plt.plot(steps, trains, label='train', alpha=0.8)
plt.plot(steps, vals, label='val', linewidth=2)
plt.axhline(math.log(VOCAB_SIZE), color='gray', linestyle=':', label=f'random (ln {VOCAB_SIZE})')
plt.xlabel('step'); plt.ylabel('cross-entropy (nats/token)')
plt.title(f'Teacher transformer (d={D_MODEL}, L={N_LAYERS}, V={VOCAB_SIZE})')
plt.legend(); plt.grid(alpha=0.3); plt.show()

# Convert val nats → BPC
sample = val_data[:5000].tolist()
avg_cpt = wz.metrics.avg_chars_per_token(tok, sample)
bpc = wz.metrics.bits_per_char(best['val'], avg_cpt)
print(f'avg chars/token: {avg_cpt:.2f}')
print(f'best val nats:   {best["val"]:.4f}')
print(f'BPC:             {bpc:.4f}')
print('(Shannon entropy of English is ~1.0 bits/char; SOTA models reach ~0.7–0.8 on Shakespeare.)')


## Cell 7 — Quick generation sample

Just to sanity-check the teacher is producing reasonable Shakespeare.


In [ ]:
for prompt in ('king', 'romeo', 'my lord,'):
    print(f'\n--- {prompt!r} ---')
    out = wz.generate.generate(
        teacher, tok,
        prompt=prompt, max_new_tokens=100,
        block_size=BLOCK_SIZE, temperature=0.7, top_k=10,
        seed=42, device=device, use_hard=False,
    )
    print(out)


## Cell 8 — Save teacher checkpoint


In [ ]:
torch.save(
    {
        'config': {
            'vocab_size': VOCAB_SIZE, 'd_model': D_MODEL, 'num_heads': NUM_HEADS,
            'n_layers': N_LAYERS, 'mlp_mult': MLP_MULT, 'block_size': BLOCK_SIZE,
        },
        'model_state': teacher.state_dict(),
        'history': history,
        'best_val': best['val'],
        'best_step': best['step'],
        'n_params': n_params,
    },
    TEACHER_PT,
)
print(f'saved teacher → {TEACHER_PT}  ({TEACHER_PT.stat().st_size/1024:.1f} KB)')
print('Download teacher.pt and bpe.json from /kaggle/working/runs/ before closing this kernel.')
